# Geometry

Representation geometry over the 20 het/hom pairs: shattering dimensionality, rank-resolved shattering, temporal generalisation and manifold capacity. Compute cells cache a summary JSON in cache/results and are skipped when it already exists; figure cells redraw from the cache.


In [ ]:
import itertools, json, sys
sys.path.append("../code")

import h5py
import matplotlib.pyplot as plt
import numpy as np
from joblib import Parallel, delayed
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

import common as C
import plots as P
import stats as st

P.apply()

_shd = C.SHD / "shd_test.h5"
if _shd.exists():
    with h5py.File(_shd, "r") as f:
        RAW = np.asarray(f["labels"][:]).astype(int)   # fine 0..19 label, cache row order
    DIGIT, LANG = RAW % 10, RAW // 10
    COND8 = 4 * LANG + 2 * (DIGIT % 2) + (DIGIT >= 5).astype(int)   # language x parity x magnitude
else:
    RAW = None
    print("shd_test.h5 not found, compute cells unavailable")


def balanced_index(cond, n_cond, cap, rng):
    # shared trial subsample, equal count per condition so chance is exact
    k = min(cap, min(int((cond == c).sum()) for c in range(n_cond)))
    idx = []
    for c in range(n_cond):
        ci = np.where(cond == c)[0]
        rng.shuffle(ci)
        idx.append(ci[:k])
    return np.sort(np.concatenate(idx)), k


def all_balanced_dichotomies(n):
    return [frozenset(p) for p in itertools.combinations(range(n), n // 2) if 0 in p]


def load_rate(seed, arch):
    d = np.load(C.repr_path(seed, arch))
    X, y = d["rate"].astype(float), d["y"].astype(int)
    assert np.array_equal(2 * (RAW // 10) + (RAW % 2), y)   # fine-label join vs cached 4-class y
    return X

## Shattering dimensionality

In [ ]:
OUT_SHAT = C.RESULTS / "shattering_n20_summary.json"
CAP_PER_COND = 100
N_SAMPLE_B = 200


def sampled_dichotomies(n, n_sample, rng):
    seen, out = set(), []
    guard = 0
    while len(out) < n_sample and guard < n_sample * 50:
        guard += 1
        plus = set(rng.choice(n, size=n // 2, replace=False).tolist())
        if 0 not in plus:
            plus = set(range(n)) - plus   # canonicalise to the side containing condition 0
        key = frozenset(plus)
        if key not in seen:
            seen.add(key)
            out.append(key)
    return out


def decodability(X, lab):
    cv = StratifiedKFold(5, shuffle=True, random_state=0)
    accs = []
    for tr, te in cv.split(X, lab):
        sc = StandardScaler().fit(X[tr])
        m = LogisticRegression(max_iter=1000, C=1.0).fit(sc.transform(X[tr]), lab[tr])
        accs.append(m.score(sc.transform(X[te]), lab[te]))
    return float(np.mean(accs))


if not OUT_SHAT.exists():
    # one shared plan (trial subsample + dichotomy list) for all 40 nets keeps the comparison paired
    sets = {"A_8cond": (COND8, 8), "B_20cond": (RAW.copy(), 20)}
    rng_t, rng_d = np.random.default_rng(0), np.random.default_rng(1)
    plans = {}
    for name, (cond, n_cond) in sets.items():
        idx, k = balanced_index(cond, n_cond, CAP_PER_COND, rng_t)
        dich = (all_balanced_dichotomies(n_cond) if n_cond <= 8
                else sampled_dichotomies(n_cond, N_SAMPLE_B, rng_d))
        plans[name] = dict(cond=cond, n_cond=n_cond, idx=idx, k=k, dich=dich)

    per = {name: {"het": {}, "hom": {}} for name in plans}
    for seed in C.SEEDS:
        for arch in ("het", "hom"):
            X = load_rate(seed, arch)
            for name, pl in plans.items():
                Xs, cs = X[pl["idx"]], pl["cond"][pl["idx"]]
                labs = [np.where(np.isin(cs, list(p)), 1, -1) for p in pl["dich"]]
                decs = np.asarray(Parallel(n_jobs=-1, prefer="threads")(
                    delayed(decodability)(Xs, lab) for lab in labs), float)
                per[name][arch][seed] = {"SD": float(decs.mean()), "dec_min": float(decs.min()),
                                         "dec_max": float(decs.max()), "dec_sd": float(decs.std(ddof=1))}
        print(f"seed {seed} done", flush=True)

    V = lambda name, arch: np.array([per[name][arch][s]["SD"] for s in C.SEEDS], float)
    triples = {name: st.paired_triple(V(name, "het"), V(name, "hom"), f"SD {name}") for name in plans}
    summary = {
        "analysis": "shattering_dimensionality_bernardi2020",
        "n_pairs": len(C.SEEDS),
        "decoder": "logistic, 5-fold CV",
        "cap_per_condition": CAP_PER_COND,
        "condition_sets": {name: {"n_conditions": pl["n_cond"], "trials_per_condition": pl["k"],
                                  "n_trials": int(pl["idx"].size), "n_dichotomies": len(pl["dich"]),
                                  "dichotomies_exhaustive": pl["n_cond"] <= 8}
                           for name, pl in plans.items()},
        "by_arch": {name: {arch: {"SD_mean": float(V(name, arch).mean()),
                                  "SD_sd": float(V(name, arch).std(ddof=1))}
                           for arch in ("het", "hom")} for name in plans},
        "paired": triples,
        "per_network": {name: {arch: {int(s): per[name][arch][s] for s in C.SEEDS}
                               for arch in ("het", "hom")} for name in plans},
    }
    json.dump(summary, open(OUT_SHAT, "w"), indent=1)
    print(f"wrote {OUT_SHAT.name}")
else:
    print(f"{OUT_SHAT.name} exists, skipping")

In [ ]:
summ = json.load(open(OUT_SHAT))
titles = {"A_8cond": "8 conditions (language x parity x magnitude)\n35 balanced dichotomies",
          "B_20cond": "20 conditions (digit x language)\n"
                      f"{summ['condition_sets']['B_20cond']['n_dichotomies']} sampled dichotomies"}

fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.8))
for ax, name in zip(axes, summ["paired"]):
    r = summ["paired"][name]
    means = [r["het_mean"], r["hom_mean"]]
    sds = [r["het_sd"], r["hom_sd"]]
    ax.bar([0, 1], means, 0.6, yerr=sds, color=[P.HET, P.HOM], capsize=4)
    ax.set_xticks([0, 1])
    ax.set_xticklabels([P.LABEL["het"], P.LABEL["hom"]])
    ax.set_ylabel("shattering dimensionality\n(mean CV decodability)")
    ax.set_title(titles[name])
    ax.set_ylim(max(0.70, min(means) - max(sds) - 0.02), 1.0)
    ax.text(0.5, 0.97, f"$\\Delta$ = {r['delta']:+.4f}   $d_z$ = {r['dz']:+.2f}\n"
                       f"perm {P.fmt_p(r['perm_p'], r['perm_floor'])}   wins {r['wins']}/{r['n']}\n"
                       "chance = 0.50",
            ha="center", va="top", transform=ax.transAxes, fontsize=8.5,
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#cccccc", alpha=0.9))
fig.suptitle("Shattering dimensionality, het vs hom (n = 20)", fontweight="bold", y=1.02)
P.save(fig, C.FIGDIR / "F4_11_shattering_dimension.png")

## Shattering vs retained rank

In [ ]:
OUT_RANK = C.RESULTS / "shattering_vs_rank_n20_summary.json"
RANKS = [1, 2, 3, 4, 6, 8, 12, 16, 24, 32, 48, 64]
N_FOLDS = 5

if not OUT_RANK.exists():
    IDX, K_PER = balanced_index(COND8, 8, 100, np.random.default_rng(0))
    CSUB = COND8[IDX]
    DICH = all_balanced_dichotomies(8)
    LABELS = [np.where(np.isin(CSUB, list(p)), 1, -1) for p in DICH]

    def net_curve(seed, arch):
        X = np.load(C.repr_path(seed, arch))["rate"].astype(float)[IDX]
        cv = StratifiedKFold(N_FOLDS, shuffle=True, random_state=0)
        acc = np.zeros((len(RANKS), len(DICH)))
        # folds on the condition labels so every fold is condition-balanced;
        # scaler and pca fitted on train trials only (label-free, one fit per fold)
        for tr, te in cv.split(X, CSUB):
            sc = StandardScaler().fit(X[tr])
            Xtr, Xte = sc.transform(X[tr]), sc.transform(X[te])
            p = PCA(n_components=max(RANKS), svd_solver="full").fit(Xtr)
            Ztr, Zte = p.transform(Xtr), p.transform(Xte)
            for di, lab in enumerate(LABELS):
                ytr, yte = lab[tr], lab[te]
                for ri, m in enumerate(RANKS):
                    clf = LogisticRegression(max_iter=1000, C=1.0).fit(Ztr[:, :m], ytr)
                    acc[ri, di] += clf.score(Zte[:, :m], yte)
        acc /= N_FOLDS
        return seed, arch, acc.mean(1)

    jobs = [(s, a) for s in C.SEEDS for a in ("het", "hom")]
    res = Parallel(n_jobs=-1, backend="loky", verbose=5)(delayed(net_curve)(s, a) for s, a in jobs)
    curves = {"het": {}, "hom": {}}
    for seed, arch, c in res:
        curves[arch][seed] = c
    M = {a: np.stack([curves[a][s] for s in C.SEEDS]) for a in ("het", "hom")}

    per_rank = {m: st.paired_triple(M["het"][:, ri], M["hom"][:, ri], f"shatter@m={m}")
                for ri, m in enumerate(RANKS)}
    sig = [m for m in RANKS if per_rank[m]["perm_p"] < 0.05 and per_rank[m]["delta"] > 0]
    onset = next((m for m in RANKS if m in sig and all(mm in sig for mm in RANKS if mm >= m)), None)

    summary = {
        "analysis": "shattering_vs_retained_rank", "n_pairs": len(C.SEEDS),
        "condition_set": {"n_conditions": 8, "trials_per_condition": K_PER,
                          "n_dichotomies": len(DICH), "exhaustive": True},
        "ranks": RANKS, "onset_rank": onset,
        "by_arch": {a: {"mean": M[a].mean(0).tolist(), "sd": M[a].std(0, ddof=1).tolist()}
                    for a in ("het", "hom")},
        "paired_by_rank": {str(m): per_rank[m] for m in RANKS},
        "per_network": {a: {int(s): curves[a][s].tolist() for s in C.SEEDS} for a in ("het", "hom")},
    }
    json.dump(summary, open(OUT_RANK, "w"), indent=1)
    print(f"wrote {OUT_RANK.name}")
else:
    print(f"{OUT_RANK.name} exists, skipping")

In [ ]:
D = json.load(open(OUT_RANK))
RANKS = D["ranks"]
seeds = sorted(int(s) for s in D["per_network"]["het"])
M = {a: np.stack([D["per_network"][a][str(s)] for s in seeds]) for a in ("het", "hom")}
delta = M["het"] - M["hom"]
dmean = delta.mean(0)
# saturation: first rank reaching 90 percent of the maximum advantage
sat = next(m for m, d in zip(RANKS, dmean) if d >= 0.9 * dmean.max())

fig, axes = plt.subplots(1, 2, figsize=(12.4, 4.8))

ax = axes[0]
for arch in ("het", "hom"):
    P.band(ax, RANKS, M[arch].mean(0), M[arch].std(0), P.COL[arch], label=P.LABEL[arch],
           marker="o", ms=3.5)
ax.axhline(0.5, color=P.CHANCE, ls=":", lw=1.2, label="chance")
ax.set_xscale("log")
ax.set_xticks(RANKS)
ax.set_xticklabels(RANKS)
ax.set_xlabel("principal components retained ($m$)")
ax.set_ylabel("shattering dimensionality")
ax.set_title("Shattering against retained rank", fontsize=10)
ax.legend(fontsize=8, loc="lower right")

ax = axes[1]
P.band(ax, RANKS, dmean, delta.std(0), P.ACCENT, marker="o", ms=3.5)
ax.axhline(0, color=P.CHANCE, ls=":", lw=1.2)
ax.axvspan(RANKS[0], sat, color=P.HET, alpha=0.07, lw=0)
ax.axvline(sat, color=P.ACCENT, ls="--", lw=1.0, alpha=0.75)
ax.set_xscale("log")
ax.set_xticks(RANKS)
ax.set_xticklabels(RANKS)
ax.set_xlabel("principal components retained ($m$)")
ax.set_ylabel("heterogeneous $-$ homogeneous")
ax.set_title("The advantage against retained rank", fontsize=10)

fig.suptitle("Shattering vs retained rank (n = 20)",
             fontweight="bold", y=1.02)
P.save(fig, C.FIGDIR / "F4_25_shattering_vs_rank.png")

## Temporal generalisation

In [ ]:
OUT_TG = C.RESULTS / "temporal_generalisation_n20_summary.json"
NPZ_TG = C.RESULTS / "temporal_generalisation_matrices.npz"
N_FOLDS = 3
CHANCE = 0.25
BIN_MS = 20


def tg_matrix(seed, arch):
    d = np.load(C.repr_path(seed, arch))
    H = d["hidden"].astype(float)   # (n_trials, 50, 64)
    y = d["y"].astype(int)
    n_bins = H.shape[1]
    M = np.zeros((n_bins, n_bins))
    cv = StratifiedKFold(N_FOLDS, shuffle=True, random_state=0)
    # scaler and decoder fitted at the train bin, applied unchanged at every test bin
    for tr, te in cv.split(H[:, 0, :], y):
        for t in range(n_bins):
            sc = StandardScaler().fit(H[tr, t, :])
            clf = LogisticRegression(max_iter=200, C=1.0).fit(sc.transform(H[tr, t, :]), y[tr])
            for tp in range(n_bins):
                M[t, tp] += clf.score(sc.transform(H[te, tp, :]), y[te])
    return seed, arch, M / N_FOLDS


def lag_profile(M):
    n = M.shape[0]
    i, j = np.indices((n, n))
    lag = np.abs(i - j)
    return np.array([M[lag == L].mean() for L in range(n)])


def summarise(M):
    n = M.shape[0]
    diag = np.diag(M)
    off = M[~np.eye(n, dtype=bool)]
    long_lag = float(lag_profile(M)[10:].mean())   # transfer beyond 200 ms
    # width: test bins holding 75 percent of the train bin's own above-chance peak
    widths = []
    for t in np.where(diag > CHANCE + 0.05)[0]:
        thresh = CHANCE + 0.75 * (diag[t] - CHANCE)
        widths.append(float((M[t] >= thresh).sum()))
    return {"diag_mean": float(diag.mean()), "diag_peak": float(diag.max()),
            "offdiag_mean": float(off.mean()),
            "gen_ratio": float((off.mean() - CHANCE) / max(diag.mean() - CHANCE, 1e-9)),
            "gen_width_bins": float(np.mean(widths)) if widths else float("nan"),
            "long_lag_acc": long_lag,
            "long_lag_ratio": float((long_lag - CHANCE) / max(diag.mean() - CHANCE, 1e-9))}


if not (OUT_TG.exists() and NPZ_TG.exists()):
    jobs = [(s, a) for s in C.SEEDS for a in ("het", "hom")]
    res = Parallel(n_jobs=-1, backend="loky", verbose=5)(delayed(tg_matrix)(s, a) for s, a in jobs)
    mats = {"het": {}, "hom": {}}
    for seed, arch, M in res:
        mats[arch][seed] = M
    per = {arch: {s: summarise(mats[arch][s]) for s in C.SEEDS} for arch in ("het", "hom")}

    V = lambda arch, k: np.array([per[arch][s][k] for s in C.SEEDS], float)
    METRICS = ["diag_mean", "offdiag_mean", "gen_ratio", "gen_width_bins",
               "long_lag_acc", "long_lag_ratio"]
    triples = {k: st.paired_triple(V("het", k), V("hom", k), k) for k in METRICS}

    n_bins = mats["het"][C.SEEDS[0]].shape[0]
    summary = {
        "analysis": "temporal_generalisation_matrix", "n_pairs": len(C.SEEDS),
        "bin_ms": BIN_MS, "n_bins": int(n_bins), "n_folds": N_FOLDS, "chance": CHANCE,
        "by_arch": {arch: {k: {"mean": float(V(arch, k).mean()), "sd": float(V(arch, k).std(ddof=1))}
                           for k in METRICS} for arch in ("het", "hom")},
        "paired": triples,
        "per_network": {arch: {int(s): per[arch][s] for s in C.SEEDS} for arch in ("het", "hom")},
    }
    json.dump(summary, open(OUT_TG, "w"), indent=1)
    np.savez_compressed(NPZ_TG,
                        het=np.mean([mats["het"][s] for s in C.SEEDS], axis=0),
                        hom=np.mean([mats["hom"][s] for s in C.SEEDS], axis=0),
                        lag_het=np.stack([lag_profile(mats["het"][s]) for s in C.SEEDS]),
                        lag_hom=np.stack([lag_profile(mats["hom"][s]) for s in C.SEEDS]))
    print(f"wrote {OUT_TG.name}, {NPZ_TG.name}")
else:
    print(f"{OUT_TG.name} exists, skipping")

In [ ]:
summ = json.load(open(OUT_TG))
Z = np.load(NPZ_TG)
Mh, Mo = Z["het"], Z["hom"]
diff = Mh - Mo
n_bins, bin_ms = summ["n_bins"], summ["bin_ms"]
ext = [0, n_bins * bin_ms, n_bins * bin_ms, 0]

fig, axes = plt.subplots(2, 2, figsize=(11.6, 10.0), constrained_layout=True)
vmin, vmax = summ["chance"], max(Mh.max(), Mo.max())
ims = []
for ax, (M, title) in zip(axes.flat[:2], [(Mh, "heterogeneous"), (Mo, "homogeneous")]):
    im = ax.imshow(M, origin="upper", extent=ext, cmap="viridis", vmin=vmin, vmax=vmax,
                   aspect="equal")
    ims.append(im)
    ax.plot([0, n_bins * bin_ms], [0, n_bins * bin_ms], color="white", lw=0.7, ls=":", alpha=0.7)
    ax.set_xlabel("test time (ms)")
    ax.set_ylabel("train time (ms)")
    ax.set_title(title, fontsize=10)
    ax.grid(False)
fig.colorbar(ims[0], ax=list(axes.flat[:2]), fraction=0.046, pad=0.02, label="decoding accuracy")

ax = axes.flat[2]
lim = float(np.abs(diff).max())
imd = ax.imshow(diff, origin="upper", extent=ext, cmap="RdBu_r", vmin=-lim, vmax=lim, aspect="equal")
ax.plot([0, n_bins * bin_ms], [0, n_bins * bin_ms], color="#333", lw=0.7, ls=":", alpha=0.7)
ax.set_xlabel("test time (ms)")
ax.set_ylabel("train time (ms)")
ax.set_title("het $-$ hom", fontsize=10)
ax.grid(False)
fig.colorbar(imd, ax=ax, fraction=0.046, pad=0.02, label="accuracy difference")

ax = axes.flat[3]
lags = np.arange(n_bins) * bin_ms
for arch, prof in [("het", Z["lag_het"]), ("hom", Z["lag_hom"])]:
    P.band(ax, lags, prof.mean(0), prof.std(0), P.COL[arch], label=P.LABEL[arch])
P.chance_line(ax)
ax.set_xlabel("train-test lag (ms)")
ax.set_ylabel("decoding accuracy")
ax.set_title("Transfer against lag", fontsize=10)
ax.legend(fontsize=8)

fig.suptitle("Temporal generalisation of the class code (mean over 20 seeds)", fontweight="bold")
P.save(fig, C.FIGDIR / "F4_22_temporal_generalisation.png")

## Manifold capacity

In [ ]:

OUT_CAP = C.RESULTS / "capacity_corr_n20_summary.json"
CAP_PER_COND = 60
N_T = 200
KAPPA = 0
N_REPS = 3
T_SEED = 0

if not OUT_CAP.exists():
    import mftma
    sets = {"A_8cond": (COND8, 8), "B_20cond": (RAW.copy(), 20)}
    rng_t = np.random.default_rng(0)
    plans = {}
    for name, (cond, n_cond) in sets.items():
        idx, k = balanced_index(cond, n_cond, CAP_PER_COND, rng_t)
        plans[name] = dict(cond=cond, n_cond=n_cond, idx=idx, k=k)

    def net_capacity(seed, arch):
        d = np.load(C.repr_path(seed, arch))
        X, y = d["rate"].astype(float), d["y"].astype(int)
        assert np.array_equal(2 * (RAW // 10) + (RAW % 2), y)
        out = {}
        for name, pl in plans.items():
            cond, idx = pl["cond"], pl["idx"]
            manifolds = [X[idx][cond[idx] == c].T for c in range(pl["n_cond"])]
            # identical t-vector seed for every net keeps the comparison paired
            out[name] = mftma.manifold_capacity_corr(manifolds, kappa=KAPPA, n_t=N_T,
                                                     n_reps=N_REPS, seed=T_SEED)
        return seed, arch, out

    jobs = [(s, a) for s in C.SEEDS for a in ("het", "hom")]
    res = Parallel(n_jobs=-1, backend="loky", verbose=5)(delayed(net_capacity)(s, a) for s, a in jobs)
    per = {name: {"het": {}, "hom": {}} for name in plans}
    for seed, arch, out in res:
        for name in plans:
            per[name][arch][seed] = out[name]

    METRICS = ["alpha_M", "R_M", "D_M", "center_corr"]
    HIB = {"alpha_M": True, "R_M": False, "D_M": False, "center_corr": False}
    V = lambda name, arch, k: np.array([per[name][arch][s][k] for s in C.SEEDS], float)
    triples = {name: {k: st.paired_triple(V(name, "het", k), V(name, "hom", k), k,
                                          higher_is_better=HIB[k]) for k in METRICS}
               for name in plans}
    summary = {
        "analysis": "manifold_capacity_corrected_cohen2020_mftma", "n_pairs": len(C.SEEDS),
        "kappa": KAPPA, "n_t": N_T, "n_reps": N_REPS, "points_per_manifold": CAP_PER_COND,
        "by_arch": {name: {arch: {k: {"mean": float(V(name, arch, k).mean()),
                                      "sd": float(V(name, arch, k).std(ddof=1))}
                                  for k in METRICS} for arch in ("het", "hom")} for name in plans},
        "paired": triples,
        "per_network": {name: {arch: {int(s): {k: per[name][arch][s][k] for k in METRICS}
                                      for s in C.SEEDS} for arch in ("het", "hom")}
                        for name in plans},
    }
    json.dump(summary, open(OUT_CAP, "w"), indent=1)
    print(f"wrote {OUT_CAP.name}")
else:
    print(f"{OUT_CAP.name} exists, skipping")

In [ ]:
from scipy.stats import pearsonr

CAPS = json.load(open(OUT_CAP))
cap20 = CAPS["per_network"]["B_20cond"]
acc = {"het": {}, "hom": {}}
for arch in ("het", "hom"):
    for s in C.SEEDS:
        d = np.load(C.repr_path(s, arch))
        acc[arch][s] = float((d["pred"] == d["y"]).mean())

xs = np.array([cap20[arch][str(s)]["alpha_M"] for arch in ("het", "hom") for s in C.SEEDS])
ys = np.array([acc[arch][s] for arch in ("het", "hom") for s in C.SEEDS])
r, p = pearsonr(xs, ys)
print(f"capacity-accuracy r = {r:.3f}, p = {p:.2e}")

fig, ax = plt.subplots(figsize=(6.6, 5))
for arch in ("het", "hom"):
    ax.scatter([cap20[arch][str(s)]["alpha_M"] for s in C.SEEDS],
               [acc[arch][s] for s in C.SEEDS],
               color=P.COL[arch], s=42, label=P.LABEL[arch], zorder=3,
               edgecolor="white", linewidth=0.5)
b1, b0 = np.polyfit(xs, ys, 1)
xx = np.linspace(xs.min(), xs.max(), 50)
ax.plot(xx, b1 * xx + b0, color=P.ACCENT, lw=1.5, ls="--", zorder=2, label=f"$r$ = {r:.2f}")
ax.set_xlabel("manifold capacity $\\alpha_M$")
ax.set_ylabel("task accuracy")
ax.set_title("Capacity vs accuracy (n = 40)")
ax.legend(loc="lower right", fontsize=9)
P.save(fig, C.FIGDIR / "F4_20_capacity_bridge.png")